# Clase 1 — Embeddings y fragmentación de texto

## De qué se trata

Un sistema RAG no “lee toda la empresa” cada vez que recibe una pregunta. Primero divide los documentos en fragmentos útiles, representa esos fragmentos como vectores y recupera solo la evidencia relevante.

**Al finalizar podrás:** explicar embeddings, elegir una métrica, fragmentar con overlap y armar un recuperador Top-K.

## Referencia visual y conceptual

Inspirado en la presentación **AEM2L1 — Embeddings y fragmentación de texto** y el diagrama **AEM2L1.excalidraw**: problema de palabras clave, cercanía semántica, similitud y chunking.

Los gráficos siguientes son reproducibles y se pueden modificar; no son capturas de las diapositivas.

## 1. El problema: palabras iguales no significan siempre lo mismo

Una búsqueda tradicional encuentra coincidencias literales. Por eso puede perder una consulta como *“restablecer red”* cuando el manual dice *“restaurar conectividad”*. También puede confundir palabras idénticas con sentidos distintos, como *banco* asiento y *banco* financiero.

| Enfoque | Encuentra paráfrasis | Conserva significado | Límite principal |
|---|---:|---:|---|
| Palabras clave | A veces | No | Depende de vocabulario exacto |
| TF-IDF | Parcialmente | Limitado | No modela bien contexto |
| Embeddings | Sí | Sí, de forma aproximada | Requiere modelo y evaluación |

In [ ]:
import matplotlib.pyplot as plt

points = {
    "reiniciar contraseña": (1.0, 1.0),
    "restablecer credenciales": (1.4, 1.3),
    "solicitar vacaciones": (4.5, 3.8),
    "facturación": (4.0, 1.2),
}
fig, ax = plt.subplots(figsize=(7, 4))
for label, (x, y) in points.items():
    ax.scatter(x, y, s=120, color="#1f4e79")
    ax.annotate(label, (x + .06, y + .08))
ax.set(title="Analogía visual de un espacio semántico", xlabel="dimensión 1", ylabel="dimensión 2")
ax.grid(alpha=.25)
plt.show()

## 2. Qué es un embedding

Un embedding es un vector denso de números. No contiene una definición legible de cada dimensión; lo importante es que textos relacionados terminan en direcciones cercanas dentro del mismo espacio vectorial.

> Regla crítica: documentos y consultas deben usar **el mismo modelo de embeddings**. Si cambia el modelo, el corpus se reindexa.

## 3. Cómo medimos cercanía

| Métrica | Fórmula conceptual | Mejor valor | Cuándo pensarla |
|---|---|---:|---|
| Coseno | Ángulo entre vectores | Mayor | El significado está en la dirección |
| Producto punto | Dirección + magnitud | Mayor | El modelo/índice lo recomienda |
| Distancia L2 | Separación espacial | Menor | El índice fue diseñado para L2 |

Cuando normalizás vectores a longitud 1, coseno y producto punto producen el mismo orden.

In [ ]:
from math import sqrt

def cosine(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = sqrt(sum(a * a for a in left))
    right_norm = sqrt(sum(b * b for b in right))
    return dot / (left_norm * right_norm)

query = [1, 2, 1]
password_chunk = [2, 4, 2]
vacation_chunk = [0, 1, 4]
print("contraseña:", round(cosine(query, password_chunk), 3))
print("vacaciones:", round(cosine(query, vacation_chunk), 3))

## 4. Chunking: la unidad que se puede recuperar

Un documento completo mezcla temas. Un chunk muy chico puede perder condiciones; uno enorme añade ruido y costo. El overlap repite una región para que una regla que cruza el borde conserve contexto.

| Decisión | Muy pequeño | Intermedio | Muy grande |
|---|---|---|---|
| Precisión | Puede perder contexto | Suele ser buen punto de partida | Mezcla temas |
| Costo por prompt | Bajo | Controlado | Alto |
| Trazabilidad | Muy granular | Manejable | Difusa |

In [ ]:
import matplotlib.pyplot as plt

words = "Para recuperar la contraseña elegí olvidé mi contraseña y revisá tu correo laboral".split()
fig, ax = plt.subplots(figsize=(10, 2.5))
for i, word in enumerate(words):
    ax.text(i, 0, word, ha="center", bbox={"boxstyle":"round","fc":"#e8eef7"})
ax.axvspan(-.5, 5.5, alpha=.25, color="#f4a261", label="chunk 1")
ax.axvspan(4.5, len(words)-.5, alpha=.25, color="#2a9d8f", label="chunk 2")
ax.set(xlim=(-1, len(words)), ylim=(-1, 1), yticks=[], title="Overlap: palabras 5 y 6 aparecen en ambos chunks")
ax.legend()
plt.show()

## Antes de ejecutar

Predicción: si aumentás el overlap, ¿qué mejora y qué empeora? Escribí una respuesta antes de modificar el código.

## Práctica guiada — crear chunks trazables

La función siguiente usa palabras para hacer visible el mecanismo. En producción podés medir tokens con el tokenizador del modelo.

In [ ]:
def split_words(text, chunk_size=8, overlap=2):
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append({
            "chunk_id": f"faq-{len(chunks)+1:03d}",
            "content": " ".join(words[start:end]),
            "start_word": start,
            "end_word": end,
            "source": "faq_document.txt",
        })
        if end == len(words):
            break
        start = end - overlap
    return chunks

text = "Las vacaciones se solicitan desde Mi tiempo y requieren aprobación del líder directo."
for chunk in split_words(text, chunk_size=7, overlap=2):
    print(chunk)

## Resultado esperado

Debés poder explicar el gráfico, relacionar el resultado con una decisión de ingeniería y modificar al menos un parámetro sin perder trazabilidad.

## Errores frecuentes

- Copiar valores o parámetros sin relacionarlos con el corpus y las consultas.
- Confundir una demo que funciona con una medición de calidad.
- Omitir IDs, fuentes, configuración o validaciones.

## Mini desafío

Aplicá la idea al FAQ de RR.HH. del proyecto integrador. Escribí una hipótesis, cambiá un parámetro y registrá qué métrica usarías para decidir si fue una mejora.

## Cierre

La próxima clase muestra cómo guardar esos vectores y buscar millones de chunks sin perder calidad.

Después continuá con los notebooks E01–E10: allí la práctica va de un ejemplo mínimo a una implementación más robusta.